<div dir="rtl">
<h1>دو ورودی برابرند یا متفاوت؟</h1>
<p>درس 23 از 76 · چرا شبکه به تابع غیرخطی نیاز دارد؟ · <code dir="ltr">19-network</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-03/chapter-03/19-network.html">📖 بازگشت به همین درس</a></p>
<p>شبکهٔ XOR و یک گام آموزش را خودتان بنویسید و نقش Activation را جدا بسنجید.</p><p>پیش‌نیاز: 17-autograd و18-module؛ Cross-Entropy و Adam در درس جاری.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>آیا زیادکردن تعداد Layer‌های Affine، بدون Activation، دو قطر مربع XOR را با یک مرز خطی جدا می‌کند؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
torch.set_num_threads(1)
torch.manual_seed(42)
x = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
targets = torch.tensor([0, 1, 1, 0], dtype=torch.long)
print("Features:", x.tolist(), "Targets:", targets.tolist())

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>build_network(width) شبکهٔ Linear(2,width)، Tanh و Linear(width,2) بسازد. train_step(Model,Optimizer,x,targets) Gradient قبلی را پاک کند، Cross-Entropy را مستقیماً از Logits محاسبه و backward کند و یک Step انجام دهد؛ Loss پیش از update را به‌صورت float برگرداند.</p>
</div>

In [ ]:
def build_network(width):
    # TODO: return the three-layer Sequential
    return None

def train_step(model, optimizer, x, targets):
    # TODO: one complete update; return pre-update loss as a float
    return None

In [ ]:
def test_exercise():
    result = build_network(8)
    if result is None:
        return False
    assert sum(p.numel() for p in result.parameters()) == 42
    assert tuple(result(x).shape) == (4, 2)
    torch.manual_seed(42)
    model = build_network(8)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
    before = [p.detach().clone() for p in model.parameters()]
    first = train_step(model, optimizer, x, targets)
    if first is None:
        return False
    assert isinstance(first, float)
    assert any(not torch.equal(a, b) for a, b in zip(before, model.parameters()))
    for _ in range(399):
        train_step(model, optimizer, x, targets)
    assert torch.equal(model(x).argmax(-1), targets)
    assert F.cross_entropy(model(x), targets).item() < first
    return True

exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: implement the TODO and rerun')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط وجود Activation را تغییر دهید و وزن‌ها را ثابت نگه دارید. خروجی در نقطهٔ میانی دو ورودی را با میانگین خروجی‌های آن‌ها مقایسه کنید؛ تبدیل Affine این برابری را حفظ می‌کند، اما تبدیل غیرخطی الزاماً نه.</p>
</div>

In [ ]:
for nonlinear in [False, True]:
    torch.manual_seed(42)
    candidate = nn.Sequential(nn.Linear(2, 8), nn.Tanh() if nonlinear else nn.Identity(), nn.Linear(8, 2))
    midpoint = (x[0]+x[3])/2
    difference = candidate(midpoint)-(candidate(x[0])+candidate(x[3]))/2
    print("Activation:", nonlinear, "Midpoint difference:", difference.detach().tolist())

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>در قرارداد هدف تک‌کلاسه، شناسه‌ها باید long باشند. تابع class_ids فقط فهرست intهای نامنفی را بپذیرد و Tensor نوع long برگرداند؛ float را با گردکردن پنهان نکنید.</p>
</div>

In [ ]:
try:
    F.cross_entropy(torch.zeros(4, 2), targets.float())
except RuntimeError as error:
    print("Expected target dtype failure:", error)
else:
    raise AssertionError("These class IDs must be integer tensors")

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def class_ids(values):
    # TODO: validate before converting
    return None

In [ ]:
def test_repair():
    result = class_ids([0, 1, 1, 0])
    if result is None:
        return False
    assert result.dtype == torch.long and torch.equal(result, targets)
    for invalid in ([0.5, 1], [True, 0], [-1]):
        try:
            class_ids(invalid)
        except ValueError:
            pass
        else:
            raise AssertionError("Do not silently round or reinterpret class IDs")
    return True

repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: implement the TODO and rerun')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>mini_gpt/experiments.py آزمایش network را با همین XOR اجرا می‌کند. FFN در mini_gpt/transformer.py نیز دو تبدیل Affine و یک Activation دارد، با ابعاد و تابع متفاوت.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>در این آزمایش کدام شاهد دربارهٔ ظرفیت است و چرا چهار پاسخ درست، شاهد تعمیم به دادهٔ ندیده نیست؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-03/chapter-03/19-network.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/19-network.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>